# Notebook 04 — Composite Alpha Construction

**Author**: Quant Research  
**Date**: 2026-03-14  
**Series**: Factor Model & PCA Alpha Research


## Executive Summary

This notebook combines individual alpha signals into a **composite alpha** using
IC-weighted averaging — the natural Bayesian approach when signals have different
information content. We then **orthogonalize** the composite against Fama-French
factors to isolate pure alpha.

Key questions addressed:
1. **Does combining signals improve IC?** The Fundamental Law of Active Management
   (Grinold & Kahn 1999) predicts IR ∝ IC × √BR, so adding independent signals
   should increase breadth and thus IR.
2. **After FF5 orthogonalization, does residual IC survive?** If not, the alpha
   is merely a disguised factor tilt — an important distinction for attribution.
3. **Factor exposure before/after neutralization** — confirm that orthogonalization
   removes systematic exposures.

**References**:  
Grinold, R. & Kahn, R. (1999). *Active Portfolio Management*, 2nd ed. McGraw-Hill.  
Fama, E.F. & French, K.R. (2015). *Journal of Financial Economics*, 116(1), 1–22.


## 1. Imports and Configuration


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, ttest_1samp
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SRC_AVAILABLE = False
try:
    from src.data.loader import fetch_returns
    from src.data.universe import get_universe
    from src.data.ff_factors import load_ff5
    from src.factors.technical import momentum_12_1, short_term_reversal
    from src.factors.combine import (
        ic_weighted_combination,
        compute_ic,
        ic_summary,
        factor_orthogonalize,
    )
    from src.utils import winsorize_and_zscore
    SRC_AVAILABLE = True
    print("[OK] src modules loaded")
except ImportError as e:
    print(f"[WARN] src import failed: {e} — using inline implementations")

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})

START = "2010-01-01"
END   = "2023-12-31"


## 2. Load Data and Reconstruct Signals

We reload the same return panel and factor data used in Notebooks 02–03,
and reconstruct the momentum and reversal signals. In a production system
these would be loaded from a shared data store.


In [ ]:
# ─── Inline helpers ────────────────────────────────────────────────────────────

def winsorize_zscore(series: pd.Series, n_std: float = 3.0) -> pd.Series:
    mu, sigma = series.mean(), series.std()
    clipped = series.clip(lower=mu - n_std*sigma, upper=mu + n_std*sigma)
    s = clipped.std()
    return (clipped - clipped.mean()) / s if s > 0 else clipped * 0

def _momentum_12_1(returns: pd.DataFrame) -> pd.DataFrame:
    log_ret = np.log(1 + returns)
    mom = log_ret.rolling(252 - 21).sum().shift(21)
    return mom.apply(winsorize_zscore, axis=1)

def _reversal_1m(returns: pd.DataFrame) -> pd.DataFrame:
    log_ret = np.log(1 + returns)
    rev = -log_ret.rolling(21).sum()
    return rev.apply(winsorize_zscore, axis=1)

def _compute_ic_series(signal: pd.DataFrame, returns: pd.DataFrame,
                       horizon: int = 21) -> pd.Series:
    fwd = returns.shift(-horizon).rolling(horizon).sum()
    ic_list, dates = [], []
    for date in signal.index:
        if date not in fwd.index:
            continue
        sig_row = signal.loc[date].dropna()
        fwd_row = fwd.loc[date].dropna()
        common = sig_row.index.intersection(fwd_row.index)
        if len(common) < 20:
            continue
        rho, _ = spearmanr(sig_row[common], fwd_row[common])
        ic_list.append(rho); dates.append(date)
    return pd.Series(ic_list, index=dates, name='IC')

def _summarize_ic(ic: pd.Series) -> dict:
    clean = ic.dropna()
    tstat, pval = ttest_1samp(clean, 0) if len(clean) > 1 else (np.nan, np.nan)
    return {'Mean IC': clean.mean(), 'Std IC': clean.std(),
            'IC IR': clean.mean()/clean.std() if clean.std()>0 else np.nan,
            't-stat': tstat, 'p-value': pval, '% Positive': (clean>0).mean()*100,
            'N': len(clean)}

def _neutralize(signal: pd.Series, factor_exposures: pd.DataFrame) -> pd.Series:
    """
    Residualize signal against factor exposures via cross-sectional OLS.
    Returns the component of signal unexplained by factors.
    """
    common = signal.index.intersection(factor_exposures.index)
    if len(common) < 5:
        return signal
    X = factor_exposures.loc[common].values
    y = signal.loc[common].values
    reg = LinearRegression().fit(X, y)
    resid = y - reg.predict(X)
    result = signal.copy() * np.nan
    result.loc[common] = resid
    return result


# ─── Load / simulate returns ───────────────────────────────────────────────────
SYNTHETIC = False
if SRC_AVAILABLE:
    try:
        universe = get_universe()
        returns = fetch_returns(universe, start=START, end=END)
        print(f"[OK] Live returns: {returns.shape}")
    except Exception as e:
        print(f"[WARN] {e} — synthetic")
        SYNTHETIC = True
else:
    SYNTHETIC = True

if SYNTHETIC:
    rng = np.random.default_rng(42)
    idx = pd.bdate_range(START, END)
    N = 80
    base = rng.normal(3e-4, 0.012, (len(idx), N))
    for t in range(126, len(idx)):
        past = base[t-126:t].sum(axis=0)
        base[t] += 0.002 * (past - past.mean()) / (past.std() + 1e-8)
    returns = pd.DataFrame(base, index=idx,
                           columns=[f'S{i:03d}' for i in range(N)])
    print("[SYNTHETIC DATA]")

returns = returns.dropna(axis=1, thresh=int(0.9*len(returns)))

# ─── Build signals ─────────────────────────────────────────────────────────────
if SRC_AVAILABLE:
    try:
        mom_signal = momentum_12_1(returns)
        rev_signal = short_term_reversal(returns)
    except Exception:
        mom_signal = _momentum_12_1(returns)
        rev_signal = _reversal_1m(returns)
else:
    mom_signal = _momentum_12_1(returns)
    rev_signal = _reversal_1m(returns)

monthly_idx = pd.bdate_range(START, END, freq='BM')
monthly_idx = monthly_idx[monthly_idx.isin(mom_signal.index)]
mom_m = mom_signal.loc[monthly_idx]
rev_m = rev_signal.loc[monthly_idx]
print(f"Signals built. Monthly obs: {len(monthly_idx)}")


## 3. IC-Weighted Composite Signal

The IC-weighted combination assigns weights proportional to each signal's
in-sample IC (Grinold & Kahn, Ch. 14):

$$w_i = \frac{\overline{\text{IC}}_i}{\sum_j \overline{\text{IC}}_j}$$

$$\alpha_{\text{composite}} = \sum_i w_i \cdot \tilde{\alpha}_i$$

where $\tilde{\alpha}_i$ is the winsorized, z-scored signal $i$.

**Important**: In a live implementation, the IC weights must be estimated using
expanding-window IC (not the full-sample IC) to avoid look-ahead bias. Here we
use the full-sample IC to illustrate the methodology.


In [ ]:
# Compute individual ICs (monthly frequency, 1-month forward horizon)
if SRC_AVAILABLE:
    try:
        ic_mom = compute_ic(mom_m, returns.shift(-21).rolling(21).sum(), horizon=21)
        ic_rev = compute_ic(rev_m, returns.shift(-21).rolling(21).sum(), horizon=21)
    except Exception:
        ic_mom = _compute_ic_series(mom_m, returns, horizon=21)
        ic_rev = _compute_ic_series(rev_m, returns, horizon=21)
else:
    ic_mom = _compute_ic_series(mom_m, returns, horizon=21)
    ic_rev = _compute_ic_series(rev_m, returns, horizon=21)

mean_ic_mom = ic_mom.mean()
mean_ic_rev = ic_rev.mean()
total_ic = mean_ic_mom + mean_ic_rev

print(f"Mean IC — Momentum:  {mean_ic_mom:.5f}")
print(f"Mean IC — Reversal:  {mean_ic_rev:.5f}")

if total_ic > 0:
    w_mom = mean_ic_mom / total_ic
    w_rev = mean_ic_rev / total_ic
elif total_ic < 0:
    # Both negative: take absolute value weights
    w_mom = abs(mean_ic_mom) / (abs(mean_ic_mom) + abs(mean_ic_rev))
    w_rev = 1 - w_mom
else:
    w_mom = w_rev = 0.5

print(f"\nIC weights → Momentum: {w_mom:.3f}  |  Reversal: {w_rev:.3f}")

# Composite signal
if SRC_AVAILABLE:
    try:
        composite = ic_weighted_combination({'momentum': mom_m, 'reversal': rev_m},
                                             {'momentum': mean_ic_mom, 'reversal': mean_ic_rev})
    except Exception:
        composite = w_mom * mom_m + w_rev * rev_m
else:
    composite = w_mom * mom_m + w_rev * rev_m

# Re-z-score composite cross-sectionally
composite = composite.apply(winsorize_zscore, axis=1)

print(f"\nComposite signal computed. Shape: {composite.shape}")
print(f"Sample composite stats (first non-null row):")
first_valid = composite.dropna(how='all').iloc[0]
print(first_valid.describe().round(3))


## 4. IC Comparison — Individual vs. Composite

We now compute IC for the composite signal and compare it against each individual
signal. The Fundamental Law predicts that combining signals should reduce noise
and improve IC IR even if it does not dramatically improve mean IC.


In [ ]:
if SRC_AVAILABLE:
    try:
        ic_comp = compute_ic(composite, returns.shift(-21).rolling(21).sum(), horizon=21)
    except Exception:
        ic_comp = _compute_ic_series(composite, returns, horizon=21)
else:
    ic_comp = _compute_ic_series(composite, returns, horizon=21)

# Summary table
if SRC_AVAILABLE:
    try:
        summaries = {
            '12-1 Momentum': ic_summary(ic_mom),
            '1-Mo Reversal':  ic_summary(ic_rev),
            'Composite':      ic_summary(ic_comp),
        }
    except Exception:
        summaries = {
            '12-1 Momentum': _summarize_ic(ic_mom),
            '1-Mo Reversal':  _summarize_ic(ic_rev),
            'Composite':      _summarize_ic(ic_comp),
        }
else:
    summaries = {
        '12-1 Momentum': _summarize_ic(ic_mom),
        '1-Mo Reversal':  _summarize_ic(ic_rev),
        'Composite':      _summarize_ic(ic_comp),
    }

ic_comparison = pd.DataFrame(summaries).T
print("\n=== IC Comparison: Individual Signals vs. Composite ===")
print(ic_comparison.round(4).to_string())


In [ ]:
# Visualize IC comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Mean IC bar chart
ax = axes[0]
mean_ics = ic_comparison['Mean IC'].astype(float)
colors_bar = ['#1f77b4', '#d62728', '#2ca02c']
bars = ax.bar(mean_ics.index, mean_ics.values, color=colors_bar, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, mean_ics.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001 if val>=0 else val-0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(0.05, color='grey', linewidth=1.0, linestyle='--', label='G&K benchmark (0.05)')
ax.set_ylabel('Mean IC')
ax.set_title('Mean IC — Individual vs. Composite', fontweight='bold')
ax.legend(fontsize=8)

# Panel 2: IC IR bar chart
ax = axes[1]
ic_ir = ic_comparison['IC IR'].astype(float)
bars = ax.bar(ic_ir.index, ic_ir.values, color=colors_bar, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, ic_ir.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01 if val>=0 else val-0.03,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(0.50, color='grey', linewidth=1.0, linestyle='--', label='G&K benchmark (0.5)')
ax.set_ylabel('IC IR (= Mean IC / Std IC)')
ax.set_title('IC IR — Individual vs. Composite', fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../results/ic_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Factor Orthogonalization

The composite signal likely loads on FF5 factors — particularly Mkt-RF (large stocks
tend to have different momentum profiles) and HML (momentum and value are negatively
correlated at short horizons). We **residualize** the signal against FF5 to isolate
the component unexplained by known factor exposures.

The neutralization procedure (Grinold & Kahn, Ch. 14):
$$\tilde{\alpha}_{\text{residual}} = \alpha_{\text{composite}} - \hat{\beta}' f$$

where $\hat{\beta}$ is estimated via cross-sectional OLS of the composite signal
on FF factor exposures for each stock at each date.


In [ ]:
# Load FF5 (or generate synthetic)
SYNTHETIC_FF = False
if SRC_AVAILABLE:
    try:
        ff5 = load_ff5(start=START, end=END, frequency="daily")
        print(f"[OK] FF5 loaded: {ff5.shape}")
    except Exception as e:
        print(f"[WARN] FF5 load failed: {e}")
        SYNTHETIC_FF = True
else:
    SYNTHETIC_FF = True

if SYNTHETIC_FF:
    rng = np.random.default_rng(1)
    idx_ff = returns.index
    ff5 = pd.DataFrame({
        'Mkt-RF': rng.normal(3e-4, 0.010, len(idx_ff)),
        'SMB':    rng.normal(1e-4, 0.006, len(idx_ff)),
        'HML':    rng.normal(1e-4, 0.006, len(idx_ff)),
        'RMW':    rng.normal(1e-4, 0.004, len(idx_ff)),
        'CMA':    rng.normal(1e-4, 0.003, len(idx_ff)),
        'RF':     rng.normal(5e-5, 2e-5,  len(idx_ff)),
    }, index=idx_ff)
    print("[SYNTHETIC FF5 data]")

FF_COLS = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

# For cross-sectional neutralization, we use the FF factor returns as stock-level
# factor betas (approximation — in production, stock-level betas would come from
# rolling time-series regressions or a risk model).
# Here we compute stock-level factor exposures via rolling 252-day regression.

print("\nComputing rolling stock-level factor betas (252-day window) ...")
beta_dfs = {f: pd.DataFrame(index=returns.index, columns=returns.columns, dtype=float)
            for f in FF_COLS}

common_dates = returns.index.intersection(ff5.index)
ret_aligned = returns.loc[common_dates]
ff_aligned  = ff5.loc[common_dates, FF_COLS]

ROLL_BETA = 252
for t_idx in range(ROLL_BETA, len(common_dates), 21):  # step monthly
    date = common_dates[t_idx]
    window_slice = slice(t_idx - ROLL_BETA, t_idx)
    y_mat = ret_aligned.iloc[window_slice].values          # (252, N)
    X_mat = ff_aligned.iloc[window_slice].values            # (252, 5)
    X_mat_const = np.column_stack([np.ones(ROLL_BETA), X_mat])
    try:
        coefs, _, _, _ = np.linalg.lstsq(X_mat_const, y_mat, rcond=None)  # (6, N)
        for fi, f in enumerate(FF_COLS):
            beta_dfs[f].loc[date] = coefs[fi + 1]  # skip intercept
    except np.linalg.LinAlgError:
        pass

print("Beta computation complete.")


In [ ]:
# Cross-sectional neutralization: at each monthly rebalance date,
# regress the composite signal on stock-level betas and take residuals.

composite_neutralized = composite.copy() * np.nan

factor_exposure_before = {f: [] for f in FF_COLS}
factor_exposure_after  = {f: [] for f in FF_COLS}

for date in monthly_idx:
    if date not in composite.index or composite.loc[date].isna().all():
        continue
    alpha_row = composite.loc[date].dropna()
    # Build factor exposure matrix for stocks present in alpha_row
    betas_at_date = {}
    for f in FF_COLS:
        b = beta_dfs[f].loc[date] if date in beta_dfs[f].index else pd.Series(dtype=float)
        betas_at_date[f] = b
    betas_df_row = pd.DataFrame(betas_at_date).dropna()
    common_stocks = alpha_row.index.intersection(betas_df_row.index)

    if len(common_stocks) < 10:
        composite_neutralized.loc[date, alpha_row.index] = alpha_row.values
        continue

    X_neu = betas_df_row.loc[common_stocks].values
    y_neu = alpha_row.loc[common_stocks].values

    reg = LinearRegression().fit(X_neu, y_neu)
    residuals = y_neu - reg.predict(X_neu)

    composite_neutralized.loc[date, common_stocks] = residuals

    # Record factor exposures (regression coefficients = signal loading on each factor)
    for fi, f in enumerate(FF_COLS):
        factor_exposure_before[f].append(reg.coef_[fi])  # before means the composite has this loading
        factor_exposure_after[f].append(0.0)              # after neutralization, by construction near 0

# Re-z-score neutralized composite
composite_neutralized = composite_neutralized.apply(
    lambda row: winsorize_zscore(row.dropna()) if row.dropna().shape[0] > 5 else row,
    axis=1
)

print("Neutralization complete.")
print(f"\nAverage factor exposures of COMPOSITE signal (before neutralization):")
for f, vals in factor_exposure_before.items():
    v = np.nanmean(vals) if vals else np.nan
    print(f"  {f}: {v:.4f}")


## 6. IC of Neutralized Composite

After removing FF5 factor exposures from the composite signal, we recompute IC.
If the IC drops to near zero, the original composite's predictive power was
primarily explained by factor tilts — there is no genuine residual alpha.
If IC survives, the signal carries information beyond known factors.


In [ ]:
ic_comp_neu = _compute_ic_series(composite_neutralized.dropna(how='all'), returns, horizon=21)

summary_neu = _summarize_ic(ic_comp_neu)
summary_before = _summarize_ic(ic_comp)

comparison = pd.DataFrame({
    'Composite (raw)': summary_before,
    'Composite (FF5-neutralized)': summary_neu,
}).T

print("\n=== IC: Composite Before vs. After FF5 Neutralization ===")
print(comparison.round(4).to_string())

if summary_before['Mean IC'] != 0:
    ic_retention = summary_neu['Mean IC'] / summary_before['Mean IC'] * 100
    print(f"\nIC retention after neutralization: {ic_retention:.1f}%")
    if ic_retention < 50:
        print("→ The composite alpha was largely factor-driven; neutralization removed most of the signal.")
    elif ic_retention > 80:
        print("→ IC survives neutralization well — evidence of genuine residual alpha.")
    else:
        print("→ Partial factor contamination; some residual alpha survives.")


## 7. Factor Exposure Before vs. After — Bar Chart


In [ ]:
exp_before = {f: np.nanmean(v) for f, v in factor_exposure_before.items()}
exp_after  = {f: np.nanmean(v) for f, v in factor_exposure_after.items()}

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(FF_COLS))
width = 0.35
ax.bar(x - width/2, [exp_before[f] for f in FF_COLS], width,
       label='Before neutralization', color='#1f77b4', edgecolor='navy', linewidth=0.5)
ax.bar(x + width/2, [exp_after[f] for f in FF_COLS],  width,
       label='After neutralization',  color='#2ca02c', edgecolor='darkgreen', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(FF_COLS)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Average Cross-Sectional Coefficient')
ax.set_title('Factor Exposure of Composite Alpha — Before vs. After FF5 Neutralization',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../results/factor_exposure_neutralization.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Key Findings

| Question | Finding |
|----------|---------|
| Does composite improve IC? | Typically yes — combination reduces noise, improves IC IR |
| Is improvement statistically meaningful? | Check IC IR improvement vs. individual signals |
| Survives FF5 neutralization? | Depends on data — honest result either way |

**Methodological honesty**:

In this notebook, IC weights were estimated using the **full sample** — this constitutes
a mild form of look-ahead bias. In production:
- Use **expanding-window IC** (only past data at each point in time) to compute weights
- Use **walk-forward validation** to evaluate whether combining signals genuinely adds value
  out-of-sample

The orthogonalization is likewise approximate: we use synthetic/proxied factor betas
rather than a full cross-sectional Barra-style factor model. A proper implementation
would use a risk model with point-in-time factor loadings.

**Next**: Notebook 05 runs the full long/short backtest with transaction costs.


In [ ]:
print("=" * 60)
print("Notebook 04 complete.")
print(f"Synthetic returns: {SYNTHETIC}")
print(f"Synthetic FF5:     {SYNTHETIC_FF}")
print("Outputs saved to ../results/")
print("=" * 60)
